# 04 Region Talk YDB bloggers importer

Stream the exact read-only YDB snapshot directly into the ACTIVE master.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '4fda410f57ccdcb0c4a1b378d5b4113fcbb8cce93de92475d4eabab7849bc9ac'
RUNTIME_CONTRACT = 'region-talk-ydb-bloggers-import.v1'
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary source for the bounded read-only YDB blogger import notebook."""\n\nfrom __future__ import annotations\n\nimport os\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom uuid import UUID\n\nimport psycopg\nimport ydb\n\nfrom my_data_hub.hashing import canonical_json_bytes\nfrom my_data_hub.workloads.bloggers.importer import BloggerSnapshotImporter\nfrom my_data_hub.workloads.bloggers.ydb_reader import YdbBloggerSnapshot\n\n\ndef _required(name: str) -> str:\n    value = os.environ.get(name, "")\n    if not value:\n        raise RuntimeError(f"required runtime value is absent: {name}")\n    return value\n\n\ndef main() -> int:\n    endpoint = _required("MY_DATA_HUB_YDB_ENDPOINT")\n    database = _required("MY_DATA_HUB_YDB_DATABASE")\n    # Credentials are supplied through Kaggle User Secrets/federation.  They are\n    # consumed by the official SDK and never serialized into the receipt.\n    driver = ydb.Driver(endpoint=endpoint, database=database, credentials=ydb.credentials_from_env_variables())\n    driver.wait(timeout=20, fail_fast=True)\n    snapshot_at = datetime.fromisoformat(_required("MY_DATA_HUB_YDB_SNAPSHOT_AT").replace("Z", "+00:00"))\n    expected = int(_required("MY_DATA_HUB_YDB_EXPECTED_ROWS"))\n    project_id = UUID(_required("MY_DATA_HUB_REGION_TALK_PROJECT_ID"))\n    try:\n        with psycopg.connect(_required("MY_DATA_HUB_MASTER_MIGRATION_URL"), connect_timeout=15) as connection:\n            with connection.cursor() as cursor:\n                cursor.execute("SET statement_timeout=\'30min\'")\n                cursor.execute("SET lock_timeout=\'5s\'")\n                cursor.execute("SET idle_in_transaction_session_timeout=\'30s\'")\n            with YdbBloggerSnapshot(driver).iter_rows() as rows:\n                receipt = BloggerSnapshotImporter().import_rows(\n                    connection,\n                    project_id=project_id,\n                    snapshot_at=snapshot_at.astimezone(UTC),\n                    expected_row_count=expected,\n                    rows=rows,\n                    source_code_revision=_required("MY_DATA_HUB_SOURCE_REVISION"),\n                )\n    finally:\n        driver.stop(timeout=5)\n    quarantined = receipt.export.dispositions.get("quarantined", 0)\n    if not receipt.accounting_complete or quarantined or receipt.export.undispositioned:\n        raise RuntimeError("blogger import accounting is not complete")\n    public = {\n        "schema_version": "region-talk-ydb-bloggers-import-receipt.v1",\n        "export_batch_id": str(receipt.export.export_batch_id),\n        "row_count": receipt.export.row_count,\n        "record_id_set_sha256": receipt.export.record_id_set_sha256,\n        "logical_sha256": receipt.export.logical_sha256,\n        "canonical_outcome_sha256": receipt.canonical_outcome_sha256,\n        "actor_count": receipt.actor_count,\n        "account_count": receipt.account_count,\n        "duplicate_group_count": receipt.duplicate_group_count,\n        "undispositioned": receipt.export.undispositioned,\n        "quarantined": quarantined,\n        "canonical_revision": receipt.canonical_revision,\n        "durability_state": receipt.durability_state,\n    }\n    Path("/kaggle/working/blogger-import-receipt.json").write_bytes(canonical_json_bytes(public))\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())